In [1]:
from src.config.config import config
from src.data_management.raw_data_handler.ccontracts_c2_handler import CContractsC2Handler
from src.data_management.raw_data_handler.tgentrades_handler import TgentradesHandler

In [2]:
# CCONTRACTS_C2
CContractsC2Handler().build_raw_data(
    columns_list=config.data_config.raw_data_config.ccontracts_c2_columns_list,
    selected_columns_dict=config.data_config.raw_data_config.ccontracts_c2_columns_selected_dict,
    file_prefix=config.data_config.raw_data_config.tgentrades_prefix,
)

AttributeError: 'RawDataConfig' object has no attribute 'first_year'

In [ ]:
# TGENTRADES
TgentradesHandler().build_data_raw(
    columns_list=config.data_config.raw_data_config.tgentrades_columns_list,
    selected_columns_dict=config.data_config.raw_data_config.tgentrades_columns_selected_dict,
    file_prefix=config.data_config.raw_data_config.tgentrades_prefix,
)


In [1]:
import pandas as pd
from pathlib import Path
from tqdm import tqdm

from enum import Enum

import typing as t

# Data types for columns
class DataType(Enum):
    DATE = 0
    DATETIME = 1
    TEXT = 2
    FLOAT = 3
    INT = 4

# Years to process
FIRST_YEAR = 2017
LAST_YEAR = 2022

Common functions

In [5]:
# Custom process
def check_is_header(series: pd.Series) -> bool:
    return series.str.match(r"^[A-Za-z_]+$").all()

def default_custom_process(df: pd.DataFrame) -> pd.DataFrame:
    is_header = check_is_header(df.iloc[0])
    if is_header:
        df = df.iloc[1:, :].reset_index(drop=True)
        
    return df

# Process paths
def get_unique_files(file_list:t.List[Path]) -> t.List[Path]:
    unique_files = {}
    for file in file_list:
        name_without_extension = "".join(file.name.split(".")[:-1])
        date = name_without_extension.split("_")[-1]  # Assuming the date is the last part of the name
        if file.name not in unique_files:
            unique_files[date] = file
    return list(unique_files.values())

# Process datetimes
def prepare_datetime(time_str: str) -> str:
    """ 
    Possible values for time_str: [
        "01/01/3000 9:00:06", 
        "01/01/3000 09:00:06", 
        "09:00:00.000000", 
        "09:00:00:000", 
        "08:00:16.002360"
    ]
    """
    time_str = time_str.split(" ")[-1]
    time_str_split = time_str.split(":")
    time_str_split[0] = f"0{time_str_split[0]}" if len(time_str_split[0]) == 1 else time_str_split[0]

    if len(time_str_split) == 4:
        time_str = f"{time_str_split[0]}:{time_str_split[1]}:{time_str_split[2]}.{time_str_split[3]}"
    elif len(time_str_split) != 3:
        raise ValueError(f"Unexpected time format: {time_str}")
    else:
        time_str = ":".join(time_str_split)

    time_str_split = time_str.split(".")
    if len(time_str_split) == 1:
        time_str = f"{time_str_split[0]}.000000"

    return time_str

# Main function to build raw data
def build_data_raw(
        columns_list:t.List[str],
        selected_columns_dict:t.Dict[str, DataType],
        file_prefix: str,
        custom_processing_func: t.Optional[t.Callable] = default_custom_process
    ) -> pd.DataFrame:

    # Data raw
    data_raw = []

    # For every year, we read each contract type
    for year in tqdm(range(FIRST_YEAR, LAST_YEAR + 1)):
        path_year = Path(f"data/source_data/{year}")

        file_list = list(path_year.glob(f"{file_prefix}_*.TXT")) + list(path_year.glob(f"{file_prefix}_*.M3"))
        unique_file_list = get_unique_files(file_list)

        for file in unique_file_list:
            if file.is_file():
                df = pd.read_csv(
                    file,
                    delimiter=";",
                    header=None,
                    dtype="string",
                )

                # Custom process for each case. We can use the default one, which checks if the first row is a header and removes it if so.
                df = custom_processing_func(df=df)

                # Columns
                total_columns = df.shape[1]
                unknown_names = [
                    f"unknown_{i+1}"
                    for i in range(max(0, total_columns - len(columns_list)))
                ]
                column_names = columns_list + unknown_names

                # Assign
                df.columns = column_names

                # Select only relevant columns
                df = df[list(selected_columns_dict.keys())]

                # IBX mask
                IBX_mask = df["ContractCode"].str.contains(
                    ("IBX"),
                    na=False
                )
                df = df[IBX_mask]

                # Convert data types
                for col, dtype in selected_columns_dict.items():
                    if dtype == DataType.DATE:
                        df[col] = pd.to_datetime(df[col], format="%Y%m%d").dt.date    
                    elif dtype == DataType.DATETIME:
                        
                        try:
                            df[col] = df[col].str.strip().str.split(" ").str[-1].apply(prepare_datetime)
                            df[col] = pd.to_datetime(df[col], format="%H:%M:%S.%f")
                        except Exception as e:
                            raise ValueError(f"Unexpected format in column {col}. file: {file.name} Error: {e}")
                    elif dtype == DataType.FLOAT:
                        df[col] = pd.to_numeric(df[col].str.replace(",", "."), downcast="float")
                    elif dtype == DataType.INT:
                        df[col] = pd.to_numeric(df[col], downcast="integer")
                
                # Metadata
                df["Year"] = year
                df["SourceFile"] = file.name
                
                # Join data_raw
                data_raw.append(df)

    # Concatenate final DataFrame
    data_raw = pd.concat(data_raw, ignore_index=True)

    # Save CSV
    output_dir = Path(f"data/raw_data")
    output_dir.mkdir(parents=True, exist_ok=True)

    output_file = output_dir / f"{file_prefix}.csv"
    data_raw.to_csv(output_file, index=False, encoding="utf-8")

    print(f"\nArchivo guardado en: {output_file}")
    print(f"Total filas finales: {len(data_raw)}")

    return data_raw

CCONTRACTS_C2

In [6]:
CCONTRACTS_C2_COLUMNS_LIST = [
    "SessionDate",                     # 1 Fecha de sesión
    "ClearingHouseCode",               # 2 Código de cámara
    "ContractCode",                    # 3 Código de contrato
    "ContractGroupCode",               # 4 Grupo del contrato
    "ContractTypeCode",                # 5 Tipo del contrato
    "StrikePrice",                     # 6 Precio de ejercicio
    "MaturityDate",                    # 7 Fecha de vencimiento
    "TradingEndDate",                  # 8 Fecha de fin de negociación
    "ExerciseUnderlyingContractCode",  # 9 Código contrato subyacente (ejercicio)
    "MarginUnderlyingContractCode",    # 10 Código contrato subyacente (garantías)
    "ArrayCode",                       # 11 Código de matriz de garantías
    "ExpiryNumber",                    # 12 Nº vencimiento de liquidación
    "OffsetNumber",                    # 13 Nº compensación
    "ExpirySpan",                      # 14 Tipo de vencimiento (S/L)
    "MaturityMonthYear",               # 15 Identificador del vencimiento
    "ISINCode"                         # 16 Código ISIN
]

CCONTRACTS_C2_COLUMNS_SELECTED_DICT = {
    "SessionDate": DataType.DATE,       # 1 Fecha de sesión
    "ContractCode": DataType.TEXT,      # 3 Código de contrato
    "StrikePrice": DataType.FLOAT,      # 6 Precio de ejercicio
    "MaturityDate": DataType.DATE,      # 7 Fecha de vencimiento
}

In [7]:
# CCONTRACTS_C2
ccontracts_df = build_data_raw(
    columns_list=CCONTRACTS_C2_COLUMNS_LIST,
    selected_columns_dict=CCONTRACTS_C2_COLUMNS_SELECTED_DICT,
    file_prefix="CCONTRACTS_C2",
    )


100%|██████████| 6/6 [02:47<00:00, 27.95s/it]



Archivo guardado en: data\raw_data\CCONTRACTS_C2.csv
Total filas finales: 2520238


In [8]:
# NAs in StrikePrice are expected, as there are some contracts (like futures) that do not have a strike price.
ccontracts_df.isna().sum()


SessionDate         0
ContractCode        0
StrikePrice     22916
MaturityDate        0
Year                0
SourceFile          0
dtype: int64

In [9]:
ccontracts_df

,SessionDate,ContractCode,StrikePrice,MaturityDate,Year,SourceFile
0,2017-03-17,FIBXM7,<NA>,2017-06-16,2017,CCONTRACTS_C2_20170317.TXT
1,2017-03-17,FIBXZ7,<NA>,2017-12-15,2017,CCONTRACTS_C2_20170317.TXT
2,2017-03-17,CIBX 8000M17,8000.0,2017-06-16,2017,CCONTRACTS_C2_20170317.TXT
3,2017-03-17,CIBX 7800Z17,7800.0,2017-12-15,2017,CCONTRACTS_C2_20170317.TXT
4,2017-03-17,CIBX 7900Z17,7900.0,2017-12-15,2017,CCONTRACTS_C2_20170317.TXT
...,...,...,...,...,...,...
2520233,2022-06-28,PIBX 9200W4N22,9200.0,2022-07-22,2022,CCONTRACTS_C2_20220628.TXT
2520234,2022-06-28,PIBX 9300W4N22,9300.0,2022-07-22,2022,CCONTRACTS_C2_20220628.TXT
2520235,2022-06-28,PIBX 9400W4N22,9400.0,2022-07-22,2022,CCONTRACTS_C2_20220628.TXT
2520236,2022-06-28,CIBX 9500W4N22,9500.0,2022-07-22,2022,CCONTRACTS_C2_20220628.TXT


TGENTRADES

In [10]:
TGENTRADES_COLUMNS_LIST = [
    "SessionDate",    # 1 Fecha de sesión
    "MarketCode",     # 2 Código de mercado
    "TradeExecID",    # 3 Número de registro de negociación
    "ContractCode",   # 4 Código de contrato
    "ExecTime",       # 5 Hora de ejecución
    "TradePrice",     # 6 Precio
    "Quantity",       # 7 Volumen
    "TradeType"       # 8 Tipo de operación
]

TGENTRADES_COLUMNS_SELECTED_DICT = {
    "SessionDate": DataType.DATE,       # 1 Fecha de sesión
    "MarketCode": DataType.TEXT,        # 2 Código de mercado
    "TradeExecID": DataType.TEXT,       # 3 Número de registro de negociación
    "ContractCode": DataType.TEXT,      # 4 Código de contrato
    "ExecTime": DataType.DATETIME,      # 5 Hora de ejecución
    "TradePrice": DataType.FLOAT,       # 6 Precio
    "Quantity": DataType.INT,           # 7 Volumen
    "TradeType": DataType.TEXT          # 8 Tipo de operación
}

In [11]:
def tgentrades_custom_process(df: pd.DataFrame) -> pd.DataFrame:
    is_header = check_is_header(df.iloc[0])
    if is_header:
        skip_column_list = [c for c, v in df.iloc[0].items() if v.lower().strip() == "secuencia"]
        df.drop(columns=skip_column_list, inplace=True)
        df = df.iloc[1:, :].reset_index(drop=True)

    return df

In [12]:
def prepare_datetime(time_str: str) -> str:
    """ 
    Possible values for time_str: [
        "01/01/3000 9:00:06", 
        "01/01/3000 09:00:06", 
        "09:00:00.000000", 
        "09:00:00:000", 
        "08:00:16.002360"
    ]
    """
    time_str = time_str.split(" ")[-1]
    time_str_split = time_str.split(":")
    time_str_split[0] = f"0{time_str_split[0]}" if len(time_str_split[0]) == 1 else time_str_split[0]

    if len(time_str_split) == 4:
        time_str = f"{time_str_split[0]}:{time_str_split[1]}:{time_str_split[2]}.{time_str_split[3]}"
    elif len(time_str_split) != 3:
        raise ValueError(f"Unexpected time format: {time_str}")
    else:
        time_str = ":".join(time_str_split)

    time_str_split = time_str.split(".")
    if len(time_str_split) == 1:
        time_str = f"{time_str_split[0]}.000000"

    return time_str

series = pd.Series(["01/01/3000 9:00:06", "01/01/3000 09:00:06", "09:00:00.000000", "09:00:00:000", "08:00:16.002360"])

series = series.str.split(" ").str[-1].apply(prepare_datetime)

display(series)
pd.to_datetime(series, format="%H:%M:%S.%f")

0    09:00:06.000000
1    09:00:06.000000
2    09:00:00.000000
3       09:00:00.000
4    08:00:16.002360
dtype: str

0   1900-01-01 09:00:06.000000
1   1900-01-01 09:00:06.000000
2   1900-01-01 09:00:00.000000
3   1900-01-01 09:00:00.000000
4   1900-01-01 08:00:16.002360
dtype: datetime64[us]

In [13]:
# TGENTRADES
tgentrades_df = build_data_raw(
    columns_list=TGENTRADES_COLUMNS_LIST,
    selected_columns_dict=TGENTRADES_COLUMNS_SELECTED_DICT,
    file_prefix="TGENTRADES",
    custom_processing_func=tgentrades_custom_process,
    )


100%|██████████| 6/6 [02:09<00:00, 21.62s/it]



Archivo guardado en: data\raw_data\TGENTRADES.csv
Total filas finales: 17197910


In [14]:

# Validation helpers and configuration
import json
import datetime as dt
from dataclasses import dataclass, asdict, field

import numpy as np

@dataclass
class ValidationConfig:
    null_threshold_mandatory: float = 0.01
    near_constant_ratio: float = 0.99
    outlier_iqr_factor: float = 3.0
    non_negative_keywords: tuple = ("price", "strike", "qty", "quantity", "amount", "volume", "vol", "count", "number")
    trading_start_time: dt.time = field(default_factory=lambda: dt.time(7, 0))
    trading_end_time: dt.time = field(default_factory=lambda: dt.time(21, 30))
    min_valid_date: dt.date = field(default_factory=lambda: dt.date(1900, 1, 1))
    max_valid_date: dt.date = field(default_factory=lambda: dt.date(2026, 2, 7))
    max_future_maturity_years: int = 10
    max_categorical_cardinality: int = 25
    outlier_min_samples: int = 50
    sample_size: int = 5

@dataclass
class ValidationIssue:
    dataframe: str
    severity: str
    message: str
    columns: t.Optional[t.List[str]] = None
    count: t.Optional[int] = None
    sample: t.Optional[pd.DataFrame] = None

class ValidationResult:
    def __init__(self, dataframe: str):
        self.dataframe = dataframe
        self.issues: t.List[ValidationIssue] = []

    def add(self, severity: str, message: str, columns: t.Optional[t.List[str]] = None, count: t.Optional[int] = None, sample: t.Optional[pd.DataFrame] = None):
        self.issues.append(ValidationIssue(
            dataframe=self.dataframe,
            severity=severity,
            message=message,
            columns=columns,
            count=count,
            sample=sample
        ))

    def summary(self) -> t.Dict[str, int]:
        return {
            "dataframe": self.dataframe,
            "pass": sum(1 for i in self.issues if i.severity == "pass"),
            "warn": sum(1 for i in self.issues if i.severity == "warn"),
            "fail": sum(1 for i in self.issues if i.severity == "fail"),
        }

    def to_frame(self) -> pd.DataFrame:
        rows = []
        for issue in self.issues:
            rows.append({
                "dataframe": issue.dataframe,
                "severity": issue.severity,
                "message": issue.message,
                "columns": ", ".join(issue.columns) if issue.columns else "",
                "count": issue.count,
                "sample": issue.sample.head(5) if isinstance(issue.sample, pd.DataFrame) else issue.sample
            })
        return pd.DataFrame(rows)

    def to_markdown(self) -> str:
        df_repr = self.to_frame()
        try:
            return df_repr.to_markdown(index=False)
        except Exception:
            return df_repr.to_string(index=False)

    def to_json(self) -> str:
        def serialize_issue(issue: ValidationIssue) -> dict:
            payload = asdict(issue)
            if isinstance(issue.sample, pd.DataFrame):
                payload["sample"] = issue.sample.head(5).to_dict(orient="records")
            return payload
        return json.dumps([serialize_issue(i) for i in self.issues], default=str, indent=2)

    def raise_if_failed(self) -> None:
        failed = [i for i in self.issues if i.severity == "fail"]
        if failed:
            messages = "; ".join(i.message for i in failed)
            raise AssertionError(f"Validation failed for {self.dataframe}: {messages}")


In [15]:

# Generic validation checks

def _sample(df: pd.DataFrame, mask: t.Optional[pd.Series], config: ValidationConfig) -> t.Optional[pd.DataFrame]:
    if mask is None:
        return None
    mask = mask.fillna(False)
    if mask.any():
        return df.loc[mask].head(config.sample_size)
    return None

def matches_dtype(series: pd.Series, expected: DataType) -> bool:
    sample = series.dropna().head(20)
    if expected == DataType.DATE:
        return pd.api.types.is_datetime64_any_dtype(series) or sample.apply(lambda x: isinstance(x, (dt.date, pd.Timestamp))).all()
    if expected == DataType.DATETIME:
        return pd.api.types.is_datetime64_any_dtype(series)
    if expected == DataType.FLOAT:
        return pd.api.types.is_float_dtype(series)
    if expected == DataType.INT:
        return pd.api.types.is_integer_dtype(series)
    if expected == DataType.TEXT:
        return pd.api.types.is_string_dtype(series) or sample.apply(lambda x: isinstance(x, str)).all()
    return True

def infer_non_negative_columns(df: pd.DataFrame, config: ValidationConfig, explicit: t.Optional[t.Iterable[str]] = None) -> t.List[str]:
    explicit = explicit or []
    cols: t.Set[str] = set(explicit)
    for col in df.select_dtypes(include=[np.number]).columns:
        name = col.lower()
        if any(keyword in name for keyword in config.non_negative_keywords):
            cols.add(col)
    return sorted(cols)

def run_generic_checks(
    df: pd.DataFrame,
    df_name: str,
    expected_schema: t.Dict[str, DataType],
    key_columns: t.Optional[t.List[str]],
    mandatory_columns: t.Optional[t.List[str]],
    nullable_columns: t.Optional[t.List[str]],
    config: t.Optional[ValidationConfig] = None,
    explicit_non_negative: t.Optional[t.Iterable[str]] = None,
) -> ValidationResult:
    config = config or ValidationConfig()
    mandatory_columns = mandatory_columns or []
    nullable_columns = nullable_columns or []
    result = ValidationResult(df_name)

    # Schema presence
    missing_cols = [c for c in expected_schema if c not in df.columns]
    if missing_cols:
        result.add("fail", f"Missing expected columns: {missing_cols}", columns=missing_cols)
    else:
        result.add("pass", "All expected columns present", columns=list(expected_schema))
    extra_cols = [c for c in df.columns if c not in expected_schema]
    if extra_cols:
        result.add("warn", f"Unexpected columns present: {extra_cols}", columns=extra_cols)

    # Dtype check
    for col, dtype in expected_schema.items():
        if col not in df.columns:
            continue
        if matches_dtype(df[col], dtype):
            result.add("pass", f"{col} dtype matches {dtype.name}", columns=[col])
        else:
            result.add(
                "fail",
                f"{col} dtype does not match expected {dtype.name}",
                columns=[col],
                sample=df[[col]].head(config.sample_size)
            )

    # Missingness
    null_ratio = df.isna().mean()
    for col in mandatory_columns:
        ratio = float(null_ratio.get(col, 0.0))
        if ratio > config.null_threshold_mandatory:
            result.add(
                "fail",
                f"{col} null ratio {ratio:.3f} above threshold {config.null_threshold_mandatory}",
                columns=[col],
                count=int(df[col].isna().sum()),
                sample=_sample(df, df[col].isna(), config)
            )
        else:
            result.add("pass", f"{col} null ratio OK ({ratio:.3f})", columns=[col])
    for col in nullable_columns:
        ratio = float(null_ratio.get(col, 0.0))
        severity = "warn" if ratio > 0.5 else "pass"
        result.add(severity, f"{col} null ratio {ratio:.3f}", columns=[col], count=int(df[col].isna().sum()))

    # Duplicates
    full_dupes = int(df.duplicated().sum())
    if full_dupes > 0:
        result.add("warn", f"{full_dupes} full-row duplicates detected", count=full_dupes, sample=_sample(df, df.duplicated(), config))
    else:
        result.add("pass", "No full-row duplicates detected")

    if key_columns:
        dup_mask = df.duplicated(subset=key_columns, keep=False)
        dup_count = int(dup_mask.sum())
        severity = "fail" if dup_count > 0 else "pass"
        msg = f"{dup_count} duplicate rows on key {key_columns}" if dup_count else "Key is unique"
        result.add(severity, msg, columns=key_columns, count=dup_count, sample=_sample(df, dup_mask, config))

    # Non-negative numeric columns
    for col in infer_non_negative_columns(df, config, explicit_non_negative):
        if col not in df.columns:
            continue
        negative_mask = df[col] < 0
        negative_mask = negative_mask.fillna(False)
        neg_count = int(negative_mask.sum())
        if neg_count > 0:
            result.add("fail", f"{col} has {neg_count} negative values", columns=[col], count=neg_count, sample=_sample(df, negative_mask, config))
        else:
            result.add("pass", f"{col} has no negative values", columns=[col])

    # Constant / near-constant columns
    for col in df.columns:
        counts = df[col].value_counts(dropna=False)
        if counts.empty:
            continue
        top_ratio = counts.iloc[0] / len(df)
        if top_ratio >= config.near_constant_ratio:
            result.add("warn", f"{col} is constant/near-constant (top ratio {top_ratio:.3f})", columns=[col], sample=counts.head().to_frame())
        else:
            result.add("pass", f"{col} shows variation (top ratio {top_ratio:.3f})", columns=[col])

    # Outliers using IQR
    for col in df.select_dtypes(include=[np.number]).columns:
        series = df[col].dropna()
        if len(series) < config.outlier_min_samples:
            continue
        q1, q3 = series.quantile(0.25), series.quantile(0.75)
        iqr = q3 - q1
        if iqr == 0:
            continue
        lower, upper = q1 - config.outlier_iqr_factor * iqr, q3 + config.outlier_iqr_factor * iqr
        mask = (df[col] < lower) | (df[col] > upper)
        count = int(mask.sum())
        if count > 0:
            result.add("warn", f"{col} has {count} potential outliers (IQR)", columns=[col], count=count, sample=_sample(df, mask, config))
        else:
            result.add("pass", f"{col} has no IQR outliers", columns=[col])

    # String normalization
    string_cols = [c for c in df.columns if pd.api.types.is_string_dtype(df[c])]
    for col in string_cols:
        ser = df[col].dropna().astype(str)
        whitespace_mask = df[col].astype(str).str.match(r"^\s") | df[col].astype(str).str.match(r".*\s$")
        whitespace_mask = whitespace_mask.fillna(False)
        empty_mask = df[col].astype(str).str.len() == 0
        empty_mask = empty_mask.fillna(False)
        if whitespace_mask.any() or empty_mask.any():
            mask = whitespace_mask | empty_mask
            result.add("warn", f"{col} contains leading/trailing whitespace or empty strings", columns=[col], count=int(mask.sum()), sample=_sample(df, mask, config))
        else:
            result.add("pass", f"{col} strings are trimmed and non-empty", columns=[col])

        # Casing consistency for low-cardinality columns
        if ser.nunique() <= config.max_categorical_cardinality:
            lower_unique = ser.str.lower().nunique()
            if lower_unique < ser.nunique():
                result.add("warn", f"{col} has casing inconsistencies", columns=[col], sample=pd.Series(ser.unique()[:config.sample_size]))
            else:
                result.add("pass", f"{col} casing consistent across {ser.nunique()} categories", columns=[col])

    # Date/time sanity (datetime columns)
    for col in df.select_dtypes(include=["datetime64[ns]", "datetime64[ns, tz]"]).columns:
        ser = df[col].dropna()
        if ser.empty:
            continue
        min_val, max_val = ser.min(), ser.max()
        if min_val.tzinfo is not None or max_val.tzinfo is not None:
            result.add("warn", f"{col} contains timezone-aware datetimes; ensure consistency", columns=[col])
        if min_val.date() < config.min_valid_date or max_val.date() > config.max_valid_date:
            result.add("warn", f"{col} outside expected date range [{config.min_valid_date}, {config.max_valid_date}]", columns=[col], sample=ser.describe())
        else:
            result.add("pass", f"{col} within expected date range", columns=[col])

    # Categorical overview (low-cardinality only)
    for col in df.columns:
        try:
            nunique = df[col].nunique(dropna=True)
        except Exception:
            continue
        if nunique <= config.max_categorical_cardinality:
            top = df[col].value_counts(dropna=False).head(config.sample_size)
            result.add("pass", f"Top categories for {col}: {top.to_dict()}", columns=[col])

    return result


In [16]:

# Dataset-specific validation logic

def validate_ccontracts_c2(df: pd.DataFrame, config: t.Optional[ValidationConfig] = None) -> ValidationResult:
    config = config or ValidationConfig()
    expected_schema = {
        "SessionDate": DataType.DATE,
        "ContractCode": DataType.TEXT,
        "StrikePrice": DataType.FLOAT,
        "MaturityDate": DataType.DATE,
        "Year": DataType.INT,
        "SourceFile": DataType.TEXT,
    }
    mandatory_columns = ["SessionDate", "ContractCode", "MaturityDate", "Year", "SourceFile"]
    nullable_columns = ["StrikePrice"]
    key_columns = ["SessionDate", "ContractCode"]

    result = run_generic_checks(
        df=df,
        df_name="CCONTRACTS_C2",
        expected_schema=expected_schema,
        key_columns=key_columns,
        mandatory_columns=mandatory_columns,
        nullable_columns=nullable_columns,
        config=config,
        explicit_non_negative=["StrikePrice"],
    )

    # Year alignment
    session_year = pd.to_datetime(df["SessionDate"]).dt.year
    year_mismatch_mask = session_year != df["Year"]
    mismatch_count = int(year_mismatch_mask.sum())
    if mismatch_count:
        result.add("fail", f"Year column mismatch on {mismatch_count} rows", columns=["SessionDate", "Year"], count=mismatch_count, sample=_sample(df, year_mismatch_mask, config))
    else:
        result.add("pass", "Year column matches SessionDate year", columns=["Year", "SessionDate"])

    # Maturity after session date
    session_dt = pd.to_datetime(df["SessionDate"])
    maturity_dt = pd.to_datetime(df["MaturityDate"])
    invalid_maturity_mask = maturity_dt < session_dt
    invalid_count = int(invalid_maturity_mask.sum())
    if invalid_count:
        result.add("fail", f"MaturityDate before SessionDate in {invalid_count} rows", columns=["MaturityDate", "SessionDate"], count=invalid_count, sample=_sample(df, invalid_maturity_mask, config))
    else:
        result.add("pass", "MaturityDate is on/after SessionDate", columns=["MaturityDate", "SessionDate"])

    # Excessively future maturity
    future_cutoff = session_dt + pd.DateOffset(years=config.max_future_maturity_years)
    far_future_mask = maturity_dt > future_cutoff
    far_future_count = int(far_future_mask.sum())
    if far_future_count:
        result.add("warn", f"{far_future_count} rows have maturity more than {config.max_future_maturity_years} years after SessionDate", columns=["MaturityDate"], count=far_future_count, sample=_sample(df, far_future_mask, config))
    else:
        result.add("pass", "Maturity horizon within configured limit", columns=["MaturityDate"])

    # Contract code pattern
    non_ibx_mask = ~df["ContractCode"].str.contains("IBX", na=False)
    non_ibx_count = int(non_ibx_mask.sum())
    if non_ibx_count:
        result.add("fail", f"{non_ibx_count} contract codes do not contain 'IBX'", columns=["ContractCode"], count=non_ibx_count, sample=_sample(df, non_ibx_mask, config))
    else:
        result.add("pass", "All contract codes contain 'IBX'", columns=["ContractCode"])

    return result


def validate_tgentrades(
    df: pd.DataFrame,
    contracts_df: t.Optional[pd.DataFrame] = None,
    config: t.Optional[ValidationConfig] = None,
) -> ValidationResult:
    config = config or ValidationConfig()
    expected_schema = {
        "SessionDate": DataType.DATE,
        "MarketCode": DataType.TEXT,
        "TradeExecID": DataType.TEXT,
        "ContractCode": DataType.TEXT,
        "ExecTime": DataType.DATETIME,
        "TradePrice": DataType.FLOAT,
        "Quantity": DataType.INT,
        "TradeType": DataType.TEXT,
        "Year": DataType.INT,
        "SourceFile": DataType.TEXT,
    }
    mandatory_columns = list(expected_schema.keys())
    nullable_columns: t.List[str] = []
    key_columns = ["SessionDate", "TradeExecID"]

    result = run_generic_checks(
        df=df,
        df_name="TGENTRADES",
        expected_schema=expected_schema,
        key_columns=key_columns,
        mandatory_columns=mandatory_columns,
        nullable_columns=nullable_columns,
        config=config,
        explicit_non_negative=["TradePrice", "Quantity"],
    )

    # Quantity strictly positive
    non_positive_qty_mask = df["Quantity"] <= 0
    non_positive_qty_mask = non_positive_qty_mask.fillna(False)
    non_positive_qty = int(non_positive_qty_mask.sum())
    if non_positive_qty:
        result.add("fail", f"{non_positive_qty} trades have non-positive quantity", columns=["Quantity"], count=non_positive_qty, sample=_sample(df, non_positive_qty_mask, config))
    else:
        result.add("pass", "Quantities are strictly positive", columns=["Quantity"])

    # ExecTime within trading hours
    exec_time = pd.to_datetime(df["ExecTime"])
    exec_time_time = exec_time.dt.time
    before_open = exec_time_time < config.trading_start_time
    after_close = exec_time_time > config.trading_end_time
    out_of_hours_mask = before_open | after_close
    out_of_hours = int(out_of_hours_mask.sum())
    if out_of_hours:
        result.add("warn", f"{out_of_hours} rows outside trading hours {config.trading_start_time}-{config.trading_end_time}", columns=["ExecTime"], count=out_of_hours, sample=_sample(df, out_of_hours_mask, config))
    else:
        result.add("pass", "ExecTime within trading hours", columns=["ExecTime"])

    # Contract code pattern
    non_ibx_mask = ~df["ContractCode"].str.contains("IBX", na=False)
    non_ibx_count = int(non_ibx_mask.sum())
    if non_ibx_count:
        result.add("fail", f"{non_ibx_count} trade rows have contract codes without 'IBX'", columns=["ContractCode"], count=non_ibx_count, sample=_sample(df, non_ibx_mask, config))
    else:
        result.add("pass", "All trade contract codes contain 'IBX'", columns=["ContractCode"])

    # Referential integrity vs contracts
    if contracts_df is not None:
        parent_keys = contracts_df[["SessionDate", "ContractCode"]].drop_duplicates()
        merged = df[["SessionDate", "ContractCode"]].merge(parent_keys, on=["SessionDate", "ContractCode"], how="left", indicator=True)
        missing_mask = merged["_merge"] == "left_only"
        missing_count = int(missing_mask.sum())
        if missing_count:
            # Map back to original df rows
            missing_pairs = merged.loc[missing_mask, ["SessionDate", "ContractCode"]].drop_duplicates()
            join_mask = df.set_index(["SessionDate", "ContractCode"]).index.isin(missing_pairs.set_index(["SessionDate", "ContractCode"]).index)
            result.add("warn", f"{missing_count} trade rows reference contracts not present in CCONTRACTS_C2", columns=["SessionDate", "ContractCode"], count=missing_count, sample=_sample(df, pd.Series(join_mask, index=df.index), config))
        else:
            result.add("pass", "All trade contracts found in CCONTRACTS_C2 for the same session", columns=["SessionDate", "ContractCode"])

    # TradeType sanity for low cardinality
    if df["TradeType"].nunique(dropna=True) <= config.max_categorical_cardinality:
        allowed = df["TradeType"].unique()
        result.add("pass", f"Observed trade types: {allowed}", columns=["TradeType"])

    return result


# Run validations
validation_config = ValidationConfig()

ccontracts_validation = validate_ccontracts_c2(ccontracts_df, config=validation_config)
print("CCONTRACTS_C2 validation summary", ccontracts_validation.summary())
print(ccontracts_validation.to_markdown())

trade_validation = validate_tgentrades(tgentrades_df, contracts_df=ccontracts_df, config=validation_config)
print("TGENTRADES validation summary", trade_validation.summary())
print(trade_validation.to_markdown())

# Raise if any hard failures
ccontracts_validation.raise_if_failed()
trade_validation.raise_if_failed()


TypeError: Invalid datetime unit in metadata string "[ns, tz]"